# Colab 02 (Full): Step LoRA Training (Inline)
- `unified_trainer.py` JSSP step 학습 워크플로우를 노트북형으로 전체화
- HF/local dataset source 지원
- resume_from_checkpoint / shuffle_data / output_dir 자동생성 / 업로드 포함


In [ ]:
!pip -q install -U unsloth
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets trl peft accelerate bitsandbytes wandb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

## 1) 설정


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import csv
import random
from functools import partial
from pathlib import Path
import math
import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, HfApi, upload_folder
from unsloth import FastLanguageModel

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'max_seq_length': 16384,
    # 'model_type': 'llama8b',
    'model_type': 'llama8b',
    # 'model_type': 'qwen25_7b',
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # LoRA
    'lora_r': 128,
    'lora_alpha': 128,
    'lora_dropout': 0.0,
    'bias': 'none',
    'use_gradient_checkpointing': 'unsloth',
    'random_state': 42,
    'use_rslora': True,
    'loftq_config': None,

    # train hparams
    'per_device_train_batch_size': 8,
    'gradient_accumulation_steps': 2,
    'per_device_eval_batch_size': 8,
    'num_train_epochs': 2,
    'learning_rate': 2e-4,
    'save_steps': 1000,
    'save_strategy': 'steps',
    'save_total_limit': 20,
    'logging_steps': 10,
    'eval_steps': 1000,
    'warmup_steps': 1000,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'linear',
    'seed': 42,
    'group_by_length': False,
    'action_loss_weight': 4.0,
    'dataset_num_proc': 16,

    # speed controls
    'enable_eval': False,
    'eval_split_mode': 'fixed_per_size',
    'eval_instances_per_size': 3,
    'eval_split_ratio': 0.05,
    'max_train_samples': None,
    # 'max_train_samples': 10000000,
    'min_train_feasible_actions': 2,
    'min_eval_feasible_actions': 1,
    'max_eval_samples': 20,
    'eval_subset_seed': 42,

    # task flags (unified_trainer.py 호환)
    'train_jssp': True,
    'train_fssp': False,
    'train_vrp_tsp': False,
    'train_knapsack': False,
    'train_binpack': False,
    'train_lm_head': False,
    'train_embed_tokens': False,

    # environment mode
    'env_mode': 'dispatch',  # serial | dispatch

    # adapter role
    'adapter_role': 'mixed',  # policy | reason | mixed
    'action_code_width': 4,
    'action_code_cap': 9999,
    'step_supervision_mode': 'action_only',  # resolved automatically from adapter_role

    # unified_trainer.py 옵션 동등화
    'step_dataset_path': None,
    'fp16': None,
    'bf16': None,
    'dataloader_num_workers': 0,
    'evaluation_strategy': 'steps',
    'load_best_model_at_end': False,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'remove_unused_columns': False,
    'report_to': 'wandb',
    'run_name': None,

    # dataset source
    'dataset_source': 'hf',
    'step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'policy_step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'policy_step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'policy_step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'reason_step_dataset_hf_repo': 'HYUNJINI/jssp_reason_step_train_all_v1',
    'reason_step_dataset_hf_file': 'train_data/jssp_step_train_reason.jsonl',
    'reason_step_dataset_local_path': '/content/jssp_step_train_reason.jsonl',
    'mixed_step_dataset_hf_repo': 'HYUNJINI/jssp_mixed_step_train_all_v1',
    'mixed_step_dataset_hf_file': 'train_data/jssp_step_train_with_reason.jsonl',
    'mixed_step_dataset_local_path': '/content/jssp_step_train_with_reason.jsonl',
    'policy_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_policy_step_train_dispatch_v1',
    'policy_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_policy_dispatch.jsonl',
    'policy_step_dataset_local_path_dispatch': '/content/jssp_step_train_policy_dispatch.jsonl',
    'reason_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_reason_step_train_dispatch_v1',
    'reason_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_reason_dispatch.jsonl',
    'reason_step_dataset_local_path_dispatch': '/content/jssp_step_train_reason_dispatch.jsonl',
    'mixed_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_mixed_step_train_dispatch_v1',
    'mixed_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_with_reason_dispatch.jsonl',
    'mixed_step_dataset_local_path_dispatch': '/content/jssp_step_train_with_reason_dispatch.jsonl',

    # data options
    # 'shuffle_data': False,
    'shuffle_data': True,
    'shuffle_seed': 42,

    # output
    'output_accord': False,
    'output_list_of_lists': False,
    # 'output_dir': None,
    'output_dir': '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128',

    # resume
    # 'resume_from_checkpoint': None,
    'resume_from_checkpoint': '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128/checkpoint-19000',

    # wandb
    'enable_wandb': False,
    'wandb_project': None,

    # optional upload
    'upload_after_train': False,
    'upload_on_save': True,
    'hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'policy_hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'reason_hf_model_repo_out': 'HYUNJINI/jssp_reason_llama8b_step_r64_ep2',
    'upload_source': 'latest_checkpoint',  # final | latest_checkpoint | checkpoint_tag | output_dir
    'checkpoint_tag': 'checkpoint-200',
}

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.1-8B-Instruct',
    'qwen25_7b': 'unsloth/Qwen3.5-9B-Base',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
    'qwen25_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
}

print(CFG)


from transformers import TrainerCallback


def resolve_task_type(cfg):
    if cfg.get('train_jssp', False):
        return 'jssp'
    if cfg.get('train_fssp', False):
        return 'fssp'
    if cfg.get('train_vrp_tsp', False):
        return 'vrp_tsp'
    if cfg.get('train_knapsack', False):
        return 'knapsack'
    if cfg.get('train_binpack', False):
        return 'binpack'
    return 'generic'


def resolve_output_dir(cfg, task_type):
    explicit = cfg.get('output_dir')
    if explicit:
        return os.path.expanduser(str(explicit))
    env_mode = str(cfg.get('env_mode', 'serial')).lower()
    adapter_role = str(cfg.get('adapter_role', 'policy')).lower()
    model_type = str(cfg.get('model_type', 'model')).lower()
    lora_r = int(cfg.get('lora_r', 0))
    run_name = cfg.get('run_name')
    if run_name:
        folder = str(run_name)
    else:
        folder = f"{task_type}_{adapter_role}_{env_mode}_{model_type}_r{lora_r}"
    return str(Path('outputs') / folder)


def resolve_upload_dir(output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
    output_dir = Path(output_dir)
    if upload_source == 'output_dir':
        return output_dir
    if upload_source == 'final':
        final_dir = output_dir / 'final'
        return final_dir if final_dir.exists() else output_dir
    checkpoints = sorted(
        [p for p in output_dir.glob('checkpoint-*') if p.is_dir()],
        key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1,
    )
    if upload_source == 'checkpoint_tag':
        if checkpoint_tag:
            tagged = output_dir / str(checkpoint_tag)
            if tagged.exists():
                return tagged
            raise FileNotFoundError(f"checkpoint_tag not found: {tagged}")
        raise ValueError('checkpoint_tag upload_source requires CFG[\'checkpoint_tag\']')
    if checkpoints:
        return checkpoints[-1]
    final_dir = output_dir / 'final'
    if final_dir.exists():
        return final_dir
    return output_dir


def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    trainable_percent = (trainable_model_params / all_model_params * 100) if all_model_params > 0 else 0.0
    print(f"총 학습가능 매개변수: {trainable_model_params:,}개")
    print(f"총 매개변수: {all_model_params:,}개")
    print(f"학습가능 매개변수 비율: {trainable_percent:.2f}%")
    return {
        'trainable_model_params': int(trainable_model_params),
        'all_model_params': int(all_model_params),
        'trainable_percent': float(trainable_percent),
    }


class UploadCheckpointToHFCallback(TrainerCallback):
    def __init__(self, repo_id, token, output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
        self.repo_id = repo_id
        self.token = token
        self.output_dir = Path(output_dir)
        self.upload_source = upload_source
        self.checkpoint_tag = checkpoint_tag
        self.api = HfApi(token=token) if token else None

    def on_save(self, args, state, control, **kwargs):
        if not self.token:
            print('UploadCheckpointToHFCallback: HF_TOKEN empty, skip upload')
            return control
        try:
            self.api.create_repo(repo_id=self.repo_id, repo_type='model', private=False, exist_ok=True)
            folder = resolve_upload_dir(
                self.output_dir,
                upload_source=self.upload_source,
                checkpoint_tag=self.checkpoint_tag or getattr(args, 'checkpoint_tag', None),
            )
            if not folder.exists():
                print(f'UploadCheckpointToHFCallback: upload folder missing -> {folder}')
                return control
            upload_folder(
                repo_id=self.repo_id,
                repo_type='model',
                folder_path=str(folder),
                token=self.token,
                commit_message=f"Auto-upload checkpoint at step {state.global_step}",
            )
            print(f'UploadCheckpointToHFCallback: uploaded {folder} -> {self.repo_id}')
        except Exception as exc:
            print(f'UploadCheckpointToHFCallback warning: {exc}')
        return control


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
HF login ready
{'max_seq_length': 16384, 'model_type': 'llama8b', 'dtype': 'bfloat16', 'load_in_4bit': True, 'lora_r': 128, 'lora_alpha': 128, 'lora_dropout': 0.0, 'bias': 'none', 'use_gradient_checkpointing': 'unsloth', 'random_state': 42, 'use_rslora': True, 'loftq_config': None, 'per_device_train_batch_size': 8, 'gradient_accumulation_steps': 2, 'per_device_eval_batch_size': 8, 'num_train_epochs': 2, 'learning_rate': 0.0002, 'save_steps': 1000, 'save_strategy': 'steps', 'save_total_limit': 20, 'logging_steps': 10, 'eval_steps': 1000, 'warmup_steps': 1000, 'optim': 'adamw_8bit', 'weight_decay': 0.01, 'lr_scheduler_type': 'linear', 'seed': 42, 'group_by_length': False, 'action_loss_weight': 4.0, 'dataset_num_proc': 16, 'enable_eval': False, 'eval_split_mode': 'fixed_per_size', 'eval_instances_per_size': 3, 'eval_split_ratio': 0.05, 'max_train_sample

## 2) 유틸 함수 (inline)


In [ ]:
import re
from typing import Any, Dict, List, Sequence

# --- Inline helpers aligned with source action-centric step supervision ---

# --- Action special-token helpers (standalone notebook) ---
ACTION_TOKEN_PREFIX = "A"
LEGACY_ACTION_TOKEN_PREFIX = "S"
ACTION_CODE_PATTERN = re.compile(r"(?:Action\s*:\s*)?<\s*[aAsS]\s*(\d+)\s*>", re.IGNORECASE)


def format_action_code(code_index: int, code_width: int = 4, prefix: str = ACTION_TOKEN_PREFIX):
    if int(code_index) < 0:
        raise ValueError(f"code_index must be non-negative, got {code_index}.")
    if int(code_width) < 1:
        raise ValueError(f"code_width must be >= 1, got {code_width}.")
    return f"<{str(prefix)}{int(code_index):0{int(code_width)}d}>"


def parse_action_code(text: str, code_width: int = 4):
    if not isinstance(text, str):
        return None
    match = ACTION_CODE_PATTERN.search(text)
    if not match:
        return None
    return format_action_code(int(match.group(1)), code_width=code_width)


def build_action_special_tokens(code_width: int = 4, code_start: int = 1, code_cap: int = 9999, prefix: str = ACTION_TOKEN_PREFIX):
    return [
        format_action_code(code_idx, code_width=code_width, prefix=prefix)
        for code_idx in range(int(code_start), int(code_cap) + 1)
    ]


def ensure_action_special_tokens(tokenizer, model=None, code_width: int = 4, code_start: int = 1, code_cap: int = 9999, prefix: str = ACTION_TOKEN_PREFIX):
    action_tokens = build_action_special_tokens(
        code_width=code_width,
        code_start=code_start,
        code_cap=code_cap,
        prefix=prefix,
    )
    existing = set(getattr(tokenizer, "additional_special_tokens", []) or [])
    missing = [tok for tok in action_tokens if tok not in existing]
    added = 0
    if missing:
        added = int(tokenizer.add_special_tokens({"additional_special_tokens": missing}))

    if model is not None:
        target_size = len(tokenizer)
        input_size = int(model.get_input_embeddings().weight.shape[0])
        if input_size != target_size:
            model.resize_token_embeddings(target_size)

    return {
        "num_added_tokens": int(added),
        "vocab_size": int(len(tokenizer)),
        "action_token_count": int(len(action_tokens)),
    }


def action_code_to_token_id(tokenizer, action_code: str) -> int:
    token_id = tokenizer.convert_tokens_to_ids(str(action_code))
    if token_id is None:
        raise ValueError(f"Tokenizer returned None token id for action_code={action_code!r}")
    token_id = int(token_id)
    unk_id = getattr(tokenizer, "unk_token_id", None)
    if unk_id is not None and token_id == int(unk_id):
        encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
        if len(encoded) == 1:
            return int(encoded[0])
        raise ValueError(
            f"Action code {action_code!r} is not a single tokenizer token. "
            "Special tokens were likely not installed correctly."
        )
    return token_id


def token_id_to_action_code(tokenizer, token_id: int, code_width: int = 4):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    if isinstance(token, bytes):
        token = token.decode("utf-8", errors="ignore")
    token_text = str(token) if token is not None else ""
    parsed = parse_action_code(token_text, code_width=code_width)
    if parsed is not None:
        return parsed
    decoded = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    return parse_action_code(decoded, code_width=code_width)


def validate_action_tokenizer_installation(tokenizer, code_width: int = 4, code_start: int = 1, code_cap: int = 9999):
    for action_code in (
        format_action_code(code_start, code_width=code_width),
        format_action_code(min(code_start + 7, code_cap), code_width=code_width),
        format_action_code(code_cap, code_width=code_width),
    ):
        token_id = action_code_to_token_id(tokenizer, action_code)
        roundtrip = token_id_to_action_code(tokenizer, token_id, code_width=code_width)
        if roundtrip != action_code:
            raise ValueError(
                f"Action token round-trip failed: action_code={action_code}, "
                f"token_id={token_id}, roundtrip={roundtrip}"
            )

ACTION_CODE_EXTRACT_RE = re.compile(r"<\s*A\s*(\d+)\s*>", re.IGNORECASE)


def _normalize_action_code(text: str, code_width: int = 4):
    match = ACTION_CODE_EXTRACT_RE.search(str(text))
    if not match:
        return None
    return f"<A{int(match.group(1)):0{int(code_width)}d}>"


def build_step_messages(example, step_supervision_mode="action_only"):
    state_text = str(example.get("state_text", ""))
    reason_input_text = str(example.get("reason_input_text", ""))
    target_text = str(example.get("target_text", ""))
    reason_target_text = str(example.get("reason_target_text", "")).strip()

    if step_supervision_mode not in {"action_only", "action_reason", "reason_only"}:
        raise ValueError(
            f"Unsupported step_supervision_mode={step_supervision_mode}. "
            "Use one of: action_only, action_reason, reason_only."
        )

    if step_supervision_mode == "reason_only":
        if not reason_input_text:
            raise ValueError("Missing 'reason_input_text' for reason dataset sample.")
        if not reason_target_text:
            raise ValueError("Missing 'reason_target_text' for reason dataset sample.")
        user_content = reason_input_text
        assistant_content = reason_target_text
        system_content = (
            "You are an expert JSSP scheduling analyst. "
            "Explain a fixed already-selected action. "
            "Primary objective context is final makespan (Cmax). "
            "Do not output a new action. "
            "Output format:\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
    elif step_supervision_mode == "action_reason":
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        target_action_reason_text = str(example.get("target_action_reason_text", "")).strip()
        if target_action_reason_text:
            assistant_content = target_action_reason_text
        else:
            assistant_content = f"{target_text}\n{reason_target_text}" if reason_target_text else target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code and explain briefly. "
            "Output format:\n"
            "<Axxxx>\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
        user_content = state_text
    else:
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        assistant_content = target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code. "
            "Always output exactly one code in this format: <Axxxx>."
        )
        user_content = state_text

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]


def _normalize_token_ids(tokenized_output: Any) -> List[int]:
    if tokenized_output is None:
        return []
    if hasattr(tokenized_output, "tolist"):
        tokenized_output = tokenized_output.tolist()
    if isinstance(tokenized_output, tuple):
        tokenized_output = list(tokenized_output)
    if isinstance(tokenized_output, list):
        if tokenized_output and isinstance(tokenized_output[0], list):
            if len(tokenized_output) != 1:
                raise ValueError("Expected a single tokenized sequence, got a batch.")
            tokenized_output = tokenized_output[0]
        return [int(x) for x in tokenized_output]
    raise TypeError(f"Unsupported tokenized output type: {type(tokenized_output)!r}")


def _collect_action_token_ids(tokenizer) -> List[int]:
    cached = getattr(tokenizer, "_cached_action_token_ids", None)
    if cached is not None:
        return list(cached)

    token_ids = []
    seen = set()
    for token in getattr(tokenizer, "additional_special_tokens", []) or []:
        parsed = _normalize_action_code(str(token))
        if parsed is None:
            continue
        try:
            token_id = action_code_to_token_id(tokenizer, str(token))
        except Exception:
            continue
        token_id = int(token_id)
        if token_id in seen:
            continue
        seen.add(token_id)
        token_ids.append(token_id)
    setattr(tokenizer, "_cached_action_token_ids", tuple(token_ids))
    return token_ids


def _find_prompt_token_count(tokenizer, prompt_messages, full_ids: Sequence[int]) -> int:
    prompt_ids = _normalize_token_ids(
        tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
        )
    )
    if list(full_ids[: len(prompt_ids)]) == prompt_ids:
        return len(prompt_ids)

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    if callable(getattr(tokenizer, "__call__", None)):
        prompt_text_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        prompt_text_ids = [int(x) for x in prompt_text_ids]
        if list(full_ids[: len(prompt_text_ids)]) == prompt_text_ids:
            return len(prompt_text_ids)

    raise ValueError("Could not align prompt and assistant token boundary for step supervision.")


def _extract_action_codes(example: Dict[str, Any]) -> List[str]:
    codes: List[str] = []
    raw_codes = example.get("action_codes")
    if isinstance(raw_codes, list):
        codes.extend(str(code) for code in raw_codes if str(code).strip())
    action_code_to_job = example.get("action_code_to_job")
    if isinstance(action_code_to_job, dict):
        codes.extend(str(code) for code in action_code_to_job.keys() if str(code).strip())
    if not codes:
        raise ValueError(
            "Missing structured action code metadata in step example. "
            "Expected `action_codes` or `action_code_to_job`."
        )
    deduped = []
    seen = set()
    for code in codes:
        parsed = _normalize_action_code(str(code))
        if parsed is None or parsed in seen:
            continue
        seen.add(parsed)
        deduped.append(parsed)
    return deduped


def build_step_supervision_example(
    example: Dict[str, Any],
    tokenizer,
    step_supervision_mode: str = "action_only",
    max_length: int | None = None,
    action_loss_weight: float = 1.0,
):
    messages = build_step_messages(
        example=example,
        step_supervision_mode=step_supervision_mode,
    )
    full_ids = _normalize_token_ids(
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
    )
    prompt_len = _find_prompt_token_count(tokenizer, messages[:-1], full_ids)
    if prompt_len >= len(full_ids):
        raise ValueError("Assistant span is empty after chat template tokenization.")

    labels = [-100] * len(full_ids)
    loss_weights = [0.0] * len(full_ids)
    attention_mask = [1] * len(full_ids)
    action_target_mask = [0] * len(full_ids)
    assistant_positions = list(range(prompt_len, len(full_ids)))

    action_token_ids = set(_collect_action_token_ids(tokenizer))
    feasible_action_ids = []
    for code in _extract_action_codes(example):
        token_id = tokenizer.convert_tokens_to_ids(str(code))
        if token_id is None:
            continue
        feasible_action_ids.append(int(token_id))
    feasible_action_ids = sorted(set(feasible_action_ids))
    action_position = None
    if step_supervision_mode in {"action_only", "action_reason"}:
        for pos in assistant_positions:
            if int(full_ids[pos]) in action_token_ids:
                action_position = pos
                break
        if action_position is None:
            raise ValueError(
                "Could not find action token inside assistant span. "
                "Check action special-token installation and step targets."
            )

    if step_supervision_mode == "action_only":
        labels[action_position] = int(full_ids[action_position])
        loss_weights[action_position] = float(max(1.0, action_loss_weight))
        action_target_mask[action_position] = 1
    elif step_supervision_mode == "action_reason":
        for pos in assistant_positions:
            labels[pos] = int(full_ids[pos])
            loss_weights[pos] = 1.0
        loss_weights[action_position] = float(max(1.0, action_loss_weight))
        action_target_mask[action_position] = 1
    else:
        for pos in assistant_positions:
            labels[pos] = int(full_ids[pos])
            loss_weights[pos] = 1.0

    if max_length is not None and int(max_length) > 0 and len(full_ids) > int(max_length):
        full_ids = full_ids[: int(max_length)]
        attention_mask = attention_mask[: int(max_length)]
        labels = labels[: int(max_length)]
        loss_weights = loss_weights[: int(max_length)]
        action_target_mask = action_target_mask[: int(max_length)]

    supervised_token_count = sum(1 for label in labels if int(label) != -100)
    if supervised_token_count <= 0:
        raise ValueError(
            "No supervised assistant tokens remain after truncation. "
            "Increase max_length or shorten prompts."
        )

    out = dict(example)
    out["text"] = ""
    out["input_ids"] = [int(x) for x in full_ids]
    out["attention_mask"] = [int(x) for x in attention_mask]
    out["labels"] = [int(x) for x in labels]
    out["loss_weights"] = [float(x) for x in loss_weights]
    out["action_target_mask"] = [int(x) for x in action_target_mask]
    out["feasible_action_ids"] = [int(x) for x in feasible_action_ids]
    out["prompt_token_count"] = int(min(prompt_len, len(full_ids)))
    out["assistant_token_count"] = int(max(0, len(full_ids) - min(prompt_len, len(full_ids))))
    out["supervised_token_count"] = int(supervised_token_count)
    return out


## 3) 학습 실행 (full)


#### 이건 데이터 전처리 및 모델 로딩

In [ ]:
from collections import Counter
import json
import os
import random
import torch
from pathlib import Path
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import TrainerCallback
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainerCallback
base_model_name = MODEL_MAP[CFG['model_type']]
print('base model:', base_model_name)

adapter_role = str(CFG.get('adapter_role', 'policy')).lower()
resolved_step_supervision_mode = {'policy': 'action_only', 'reason': 'reason_only', 'mixed': 'action_reason'}.get(adapter_role)
if resolved_step_supervision_mode is None:
    raise ValueError(f"Unsupported CFG['adapter_role']={adapter_role}")
CFG['step_supervision_mode'] = resolved_step_supervision_mode
if adapter_role == 'policy':
    CFG['hf_model_repo_out'] = CFG.get('policy_hf_model_repo_out', CFG['hf_model_repo_out'])
elif adapter_role == 'reason':
    CFG['hf_model_repo_out'] = CFG.get('reason_hf_model_repo_out', CFG['hf_model_repo_out'])
print('adapter_role:', adapter_role)
print('resolved step supervision mode:', resolved_step_supervision_mode)

task_type = resolve_task_type(CFG)
output_dir = Path(resolve_output_dir(CFG, task_type))
output_dir.mkdir(parents=True, exist_ok=True)

if CFG.get('step_dataset_path'):
    step_dataset_path = os.path.expanduser(CFG['step_dataset_path'])
else:
    env_mode = str(CFG.get('env_mode', 'serial')).lower()
    env_suffix = '_dispatch' if env_mode == 'dispatch' else ''
    if adapter_role == 'reason':
        dataset_repo = CFG.get(f'reason_step_dataset_hf_repo{env_suffix}', CFG.get('reason_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'reason_step_dataset_hf_file{env_suffix}', CFG.get('reason_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'reason_step_dataset_local_path{env_suffix}', CFG.get('reason_step_dataset_local_path', CFG['step_dataset_local_path']))
    elif adapter_role == 'mixed':
        dataset_repo = CFG.get(f'mixed_step_dataset_hf_repo{env_suffix}', CFG.get('mixed_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'mixed_step_dataset_hf_file{env_suffix}', CFG.get('mixed_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'mixed_step_dataset_local_path{env_suffix}', CFG.get('mixed_step_dataset_local_path', CFG['step_dataset_local_path']))
    else:
        dataset_repo = CFG.get(f'policy_step_dataset_hf_repo{env_suffix}', CFG.get('policy_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'policy_step_dataset_hf_file{env_suffix}', CFG.get('policy_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'policy_step_dataset_local_path{env_suffix}', CFG.get('policy_step_dataset_local_path', CFG['step_dataset_local_path']))

    if CFG['dataset_source'] == 'hf':
        step_dataset_path = hf_hub_download(
            repo_id=dataset_repo,
            repo_type='dataset',
            filename=dataset_file,
            token=HF_TOKEN,
        )
    elif CFG['dataset_source'] == 'local':
        step_dataset_path = dataset_local_path
    else:
        raise ValueError("CFG['dataset_source'] must be 'hf' or 'local'")

print('step dataset:', step_dataset_path)

dtype = torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CFG['max_seq_length'],
    load_in_4bit=CFG['load_in_4bit'],
    dtype=dtype,
    local_files_only=False,
)

token_install = ensure_action_special_tokens(
    tokenizer=tokenizer,
    model=model,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
validate_action_tokenizer_installation(
    tokenizer=tokenizer,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
print('action token install:', token_install)

modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
if CFG['train_lm_head']:
    modules.append('lm_head')
if CFG['train_embed_tokens']:
    modules.append('embed_tokens')
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG['lora_r'],
    target_modules=modules,
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    bias=CFG['bias'],
    use_rslora=CFG['use_rslora'],
    use_gradient_checkpointing=CFG['use_gradient_checkpointing'],
    random_state=CFG['random_state'],
    loftq_config=CFG['loftq_config'],
)
print_number_of_trainable_model_parameters(model)

from datasets import Dataset
import json
import numpy as np

REQUIRED_STEP_KEYS = [
    'instance_id',
    'source_index',
    'state_text',
    'reason_input_text',
    'target_text',
    'target_action_reason_text',
    'target_reason_text',
    'reason_target_text',
    'feature_schema_version',
    'num_jobs',
    'num_machines',
    'total_steps',
    'step_idx',
    'action_codes',
    'feasible_action_codes',
    'action_code_to_job',
]


def _normalize_step_row(row):
    out = {}
    out['instance_id'] = str(row.get('instance_id', '') or '')
    source_index = row.get('source_index', -1)
    out['source_index'] = int(source_index) if source_index is not None else -1
    out['state_text'] = str(row.get('state_text', ''))
    out['reason_input_text'] = str(row.get('reason_input_text', ''))
    out['target_text'] = str(row.get('target_text', ''))
    out['target_action_reason_text'] = str(row.get('target_action_reason_text', ''))
    out['target_reason_text'] = str(row.get('target_reason_text', ''))
    out['reason_target_text'] = str(row.get('reason_target_text', ''))
    out['feature_schema_version'] = str(row.get('feature_schema_version', 'unknown'))
    out['num_jobs'] = int(row.get('num_jobs', 0))
    out['num_machines'] = int(row.get('num_machines', 0))
    out['total_steps'] = int(row.get('total_steps', 0))
    out['step_idx'] = int(row.get('step_idx', 0))

    raw_action_codes = row.get('feasible_action_codes')
    if not isinstance(raw_action_codes, list) or not raw_action_codes:
        raise ValueError("Missing non-empty 'feasible_action_codes' in step row.")
    out['action_codes'] = [str(x) for x in raw_action_codes if str(x).strip()]
    if not out['action_codes']:
        raise ValueError("Normalized 'feasible_action_codes' became empty in step row.")
    out['feasible_action_codes'] = list(out['action_codes'])
    out['num_feasible_actions'] = int(len(out['action_codes']))

    raw_action_code_to_job = row.get('action_code_to_job')
    if not isinstance(raw_action_code_to_job, dict) or not raw_action_code_to_job:
        raise ValueError("Missing non-empty 'action_code_to_job' in step row.")
    out['action_code_to_job'] = {str(k): int(v) for k, v in raw_action_code_to_job.items()}
    out['raw_feasible_action_codes'] = list(out['action_codes'])
    return out


def _scan_raw_feasible_stats(path):
    path = os.path.expanduser(str(path))
    total_rows = 0
    feasible_counter = Counter()
    with open(path, 'r', encoding='utf-8') as f:
        first_non_ws = None
        while True:
            ch = f.read(1)
            if ch == '':
                break
            if not ch.isspace():
                first_non_ws = ch
                break
        f.seek(0)

        if first_non_ws == '[':
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError('JSON input must be a list when using array format.')
            for row in data:
                normalized = _normalize_step_row(row)
                total_rows += 1
                feasible_counter[int(normalized.get('num_feasible_actions', 0))] += 1
        else:
            for line_idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except Exception as e:
                    raise ValueError(f'Invalid JSONL at line {line_idx}: {e}')
                normalized = _normalize_step_row(row)
                total_rows += 1
                feasible_counter[int(normalized.get('num_feasible_actions', 0))] += 1
    return total_rows, feasible_counter

def _iter_step_rows(path, row_cap=None, min_feasible_actions=1):
    path = os.path.expanduser(str(path))
    emitted = 0
    row_cap = None if row_cap is None else int(row_cap)
    min_feasible_actions = max(1, int(min_feasible_actions))
    with open(path, 'r', encoding='utf-8') as f:
        first_non_ws = None
        while True:
            ch = f.read(1)
            if ch == '':
                break
            if not ch.isspace():
                first_non_ws = ch
                break
        f.seek(0)

        if first_non_ws == '[':
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError('JSON input must be a list when using array format.')
            for row in data:
                normalized = _normalize_step_row(row)
                if int(normalized.get('num_feasible_actions', 0)) < min_feasible_actions:
                    continue
                yield normalized
                emitted += 1
                if row_cap is not None and emitted >= row_cap:
                    break
        else:
            for line_idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except Exception as e:
                    raise ValueError(f'Invalid JSONL at line {line_idx}: {e}')
                normalized = _normalize_step_row(row)
                if int(normalized.get('num_feasible_actions', 0)) < min_feasible_actions:
                    continue
                yield normalized
                emitted += 1
                if row_cap is not None and emitted >= row_cap:
                    break


def _try_get_source_index_array(ds):
    if 'source_index' not in set(ds.column_names):
        return None
    arrow_table = getattr(ds, '_data', None)
    if arrow_table is not None and hasattr(arrow_table, 'column'):
        try:
            column = arrow_table.column('source_index')
            chunks = [
                np.asarray(chunk.to_numpy(zero_copy_only=False), dtype=np.int64)
                for chunk in getattr(column, 'chunks', [])
            ]
            if chunks:
                return np.concatenate(chunks)
        except Exception:
            pass
    return np.asarray(ds['source_index'], dtype=np.int64)


def _try_get_int_array(ds, column_name):
    if column_name not in set(ds.column_names):
        return None
    arrow_table = getattr(ds, '_data', None)
    if arrow_table is not None and hasattr(arrow_table, 'column'):
        try:
            column = arrow_table.column(column_name)
            chunks = [
                np.asarray(chunk.to_numpy(zero_copy_only=False), dtype=np.int64)
                for chunk in getattr(column, 'chunks', [])
            ]
            if chunks:
                return np.concatenate(chunks)
        except Exception:
            pass
    return np.asarray(ds[column_name], dtype=np.int64)


def _resolve_instance_keys(ds):
    columns = set(ds.column_names)
    if 'source_index' in columns:
        source_values = _try_get_source_index_array(ds)
        if source_values is not None:
            return [f"source_{int(x)}" for x in source_values.tolist()]

    instance_values = ds['instance_id'] if 'instance_id' in columns else None
    if instance_values is None:
        raise ValueError("Step dataset must contain either 'instance_id' or 'source_index' for instance-level split.")
    keys = []
    for raw_instance_id in instance_values:
        instance_id = '' if raw_instance_id is None else str(raw_instance_id).strip()
        if not instance_id:
            raise ValueError('Empty instance_id encountered and source_index is unavailable for instance-level split.')
        keys.append(instance_id)
    return keys


def _ordered_unique(keys):
    seen = set()
    ordered = []
    for key in keys:
        if key in seen:
            continue
        seen.add(key)
        ordered.append(key)
    return ordered


def _split_indices_by_instance(
    ds,
    test_ratio,
    split_seed,
    enable_eval_split,
    split_mode='fixed_per_size',
    eval_instances_per_size=3,
):
    source_values = _try_get_source_index_array(ds)
    if source_values is not None and source_values.size > 0:
        ordered_pos = np.sort(np.unique(source_values, return_index=True)[1])
        ordered_sources = source_values[ordered_pos].tolist()
        ordered_instances = [f"source_{int(x)}" for x in ordered_sources]
        if not enable_eval_split:
            train_indices = np.arange(len(source_values), dtype=np.int64).tolist()
            return train_indices, [], ordered_instances, []

        resolved_split_mode = str(split_mode).lower()
        rng = random.Random(int(split_seed))
        if resolved_split_mode == 'fixed_per_size':
            num_jobs_values = _try_get_int_array(ds, 'num_jobs')
            num_machines_values = _try_get_int_array(ds, 'num_machines')
            if num_jobs_values is None or num_machines_values is None:
                raise ValueError("fixed_per_size eval split requires 'num_jobs' and 'num_machines' columns.")
            ordered_num_jobs = num_jobs_values[ordered_pos].tolist()
            ordered_num_machines = num_machines_values[ordered_pos].tolist()
            size_to_sources = {}
            ordered_sizes = []
            for source_idx, n_jobs, n_machines in zip(ordered_sources, ordered_num_jobs, ordered_num_machines):
                size_key = (int(n_jobs), int(n_machines))
                if size_key not in size_to_sources:
                    size_to_sources[size_key] = []
                    ordered_sizes.append(size_key)
                size_to_sources[size_key].append(int(source_idx))
            eval_sources = []
            per_size = max(1, int(eval_instances_per_size))
            for size_key in ordered_sizes:
                candidates = list(size_to_sources[size_key])
                rng.shuffle(candidates)
                eval_sources.extend(candidates[: min(per_size, len(candidates))])
        else:
            shuffled_sources = ordered_sources[:]
            rng.shuffle(shuffled_sources)
            eval_instance_count = max(1, int(round(len(shuffled_sources) * float(test_ratio))))
            eval_instance_count = min(eval_instance_count, max(1, len(shuffled_sources) - 1))
            eval_sources = shuffled_sources[:eval_instance_count]

        eval_source_set = {int(x) for x in eval_sources}
        eval_mask = np.isin(source_values, np.asarray(eval_sources, dtype=np.int64))
        train_indices = np.flatnonzero(~eval_mask).astype(np.int64).tolist()
        eval_indices = np.flatnonzero(eval_mask).astype(np.int64).tolist()
        train_instance_ids = [f"source_{int(x)}" for x in ordered_sources if int(x) not in eval_source_set]
        eval_instance_ids = [f"source_{int(x)}" for x in ordered_sources if int(x) in eval_source_set]
        return train_indices, eval_indices, train_instance_ids, eval_instance_ids

    instance_keys = _resolve_instance_keys(ds)
    ordered_instances = _ordered_unique(instance_keys)
    if not ordered_instances:
        raise ValueError('No instances found for instance-level split.')

    if not enable_eval_split:
        return list(range(len(ds))), [], ordered_instances, []

    shuffled_instances = ordered_instances[:]
    rng = random.Random(int(split_seed))
    rng.shuffle(shuffled_instances)
    eval_instance_count = max(1, int(round(len(shuffled_instances) * float(test_ratio))))
    eval_instance_count = min(eval_instance_count, max(1, len(shuffled_instances) - 1))
    eval_instance_set = set(shuffled_instances[:eval_instance_count])

    train_indices = []
    eval_indices = []
    for row_idx, instance_key in enumerate(instance_keys):
        if instance_key in eval_instance_set:
            eval_indices.append(row_idx)
        else:
            train_indices.append(row_idx)

    train_instance_ids = [iid for iid in ordered_instances if iid not in eval_instance_set]
    eval_instance_ids = [iid for iid in ordered_instances if iid in eval_instance_set]
    return train_indices, eval_indices, train_instance_ids, eval_instance_ids


def _filter_by_min_feasible_actions(ds, min_count, split_name):
    min_count = max(1, int(min_count))
    if ds is None or min_count <= 1:
        return ds
    before_count = len(ds)
    ds = ds.filter(
        lambda example: int(example.get('num_feasible_actions', len(example.get('action_codes', [])))) >= min_count,
        num_proc=max(1, int(CFG.get('dataset_num_proc', 1))),
    )
    after_count = len(ds)
    print(f'{split_name} feasible-action filter: min_feasible_actions>={min_count}, kept={after_count:,}/{before_count:,}')
    return ds


enable_eval = bool(CFG.get('enable_eval', True))
train_min_feasible_actions = max(1, int(CFG.get('min_train_feasible_actions', 1)))
eval_min_feasible_actions = max(1, int(CFG.get('min_eval_feasible_actions', 1)))
raw_train_filter_applied = False
raw_load_cap = None
raw_load_min_feasible_actions = 1
if not enable_eval:
    raw_load_min_feasible_actions = train_min_feasible_actions
    raw_train_filter_applied = raw_load_min_feasible_actions > 1
    if raw_train_filter_applied:
        print(f'raw train feasible filter enabled (no eval): min_feasible_actions>={raw_load_min_feasible_actions}')
if (not enable_eval) and CFG.get('max_train_samples') is not None:
    raw_load_cap = int(CFG['max_train_samples'])
    print(f'raw dataset load cap enabled (no eval): {raw_load_cap:,}')

raw_source_total_rows = None
raw_source_feasible_counter = None
if raw_train_filter_applied:
    print('scanning original raw dataset before feasible filter ...')
    raw_source_total_rows, raw_source_feasible_counter = _scan_raw_feasible_stats(step_dataset_path)
    raw_source_ge2 = sum(v for k, v in raw_source_feasible_counter.items() if int(k) >= 2)
    raw_source_ge3 = sum(v for k, v in raw_source_feasible_counter.items() if int(k) >= 3)
    print(f'original raw dataset rows (before feasible filter): {raw_source_total_rows:,}')
    print('original raw feasible_count_dist_top:', raw_source_feasible_counter.most_common(20))
    print('original raw feasible_count>=2 rows:', raw_source_ge2)
    print('original raw feasible_count>=2 ratio:', raw_source_ge2 / max(1, raw_source_total_rows))
    print('original raw feasible_count>=3 rows:', raw_source_ge3)
    print('original raw feasible_count>=3 ratio:', raw_source_ge3 / max(1, raw_source_total_rows))

dataset = Dataset.from_generator(
    _iter_step_rows,
    gen_kwargs={
        'path': step_dataset_path,
        'row_cap': raw_load_cap,
        'min_feasible_actions': raw_load_min_feasible_actions,
    },
)
print(f'raw dataset rows: {len(dataset):,}')

raw_feasible_counter = Counter(int(x) for x in dataset['num_feasible_actions'])
raw_ge2 = sum(v for k, v in raw_feasible_counter.items() if int(k) >= 2)
raw_ge3 = sum(v for k, v in raw_feasible_counter.items() if int(k) >= 3)
print('raw feasible_count_dist_top:', raw_feasible_counter.most_common(20))
print('raw feasible_count>=2 rows:', raw_ge2)
print('raw feasible_count>=2 ratio:', raw_ge2 / max(1, len(dataset)))
print('raw feasible_count>=3 rows:', raw_ge3)
print('raw feasible_count>=3 ratio:', raw_ge3 / max(1, len(dataset)))
if raw_source_total_rows is not None:
    print('raw feasible filter kept rows:', f"{len(dataset):,}/{raw_source_total_rows:,}")
    print('raw feasible filter kept ratio:', len(dataset) / max(1, raw_source_total_rows))
    print('raw feasible filter dropped rows:', raw_source_total_rows - len(dataset))
    print('raw feasible filter dropped ratio:', (raw_source_total_rows - len(dataset)) / max(1, raw_source_total_rows))

test_size = max(0.0, min(0.99, float(CFG.get('eval_split_ratio', 0.05))))
train_indices, eval_indices, train_instance_ids, eval_instance_ids = _split_indices_by_instance(
    dataset,
    test_ratio=test_size,
    split_seed=CFG['seed'],
    enable_eval_split=enable_eval,
    split_mode=CFG.get('eval_split_mode', 'fixed_per_size'),
    eval_instances_per_size=CFG.get('eval_instances_per_size', 3),
)

train_dataset = dataset.select(train_indices)
eval_dataset = dataset.select(eval_indices) if enable_eval else None

overlap_count = len(set(train_instance_ids) & set(eval_instance_ids))
print(
    'instance split:',
    f"mode={CFG.get('eval_split_mode', 'fixed_per_size')}",
    f"train_instances={len(train_instance_ids):,}",
    f"eval_instances={len(eval_instance_ids):,}",
    f"overlap={overlap_count}",
)
print(
    'row counts before map:',
    f"train_rows={len(train_dataset):,}",
    f"eval_rows={(len(eval_dataset) if eval_dataset is not None else 0):,}",
)

train_feasible_counter_before_filter = Counter(int(x) for x in train_dataset['num_feasible_actions'])
train_ge2_before_filter = sum(v for k, v in train_feasible_counter_before_filter.items() if int(k) >= 2)
train_ge3_before_filter = sum(v for k, v in train_feasible_counter_before_filter.items() if int(k) >= 3)
print('train feasible_count_dist_top (before feasible filter):', train_feasible_counter_before_filter.most_common(20))
print('train feasible_count>=2 rows (before feasible filter):', train_ge2_before_filter)
print('train feasible_count>=2 ratio (before feasible filter):', train_ge2_before_filter / max(1, len(train_dataset)))
print('train feasible_count>=3 rows (before feasible filter):', train_ge3_before_filter)
print('train feasible_count>=3 ratio (before feasible filter):', train_ge3_before_filter / max(1, len(train_dataset)))

if raw_train_filter_applied:
    print(f'train feasible-action filter already applied at raw load stage: min_feasible_actions>={train_min_feasible_actions}, rows={len(train_dataset):,}')
elif train_min_feasible_actions > 1:
    train_dataset = _filter_by_min_feasible_actions(train_dataset, train_min_feasible_actions, 'train')

if eval_dataset is not None and eval_min_feasible_actions > 1:
    eval_dataset = _filter_by_min_feasible_actions(eval_dataset, eval_min_feasible_actions, 'eval')

if CFG['shuffle_data']:
    print(f"shuffle train split enabled (seed={CFG['shuffle_seed']})")
    train_dataset = train_dataset.shuffle(seed=CFG['shuffle_seed'])
else:
    print('shuffle train split disabled')

if CFG.get('max_train_samples') is not None and raw_load_cap is None:
    train_cap = min(int(CFG['max_train_samples']), len(train_dataset))
    print('train sample cap:', train_cap)
    train_dataset = train_dataset.select(range(train_cap))
elif CFG.get('max_train_samples') is not None:
    print('train sample cap already applied at raw load stage:', len(train_dataset))

if eval_dataset is not None and CFG.get('max_eval_samples') is not None:
    eval_cap = min(int(CFG['max_eval_samples']), len(eval_dataset))
    print('eval sample cap:', eval_cap)
    eval_dataset = eval_dataset.shuffle(seed=int(CFG.get('eval_subset_seed', 42))).select(range(eval_cap))

fmt = partial(
    build_step_supervision_example,
    tokenizer=tokenizer,
    step_supervision_mode=resolved_step_supervision_mode,
    max_length=CFG['max_seq_length'],
    action_loss_weight=float(CFG.get('action_loss_weight', 4.0)),
)
step_map_num_proc = max(1, int(CFG.get('dataset_num_proc', 1)))
print('step supervision map num_proc:', step_map_num_proc)
train_dataset = train_dataset.map(fmt, num_proc=step_map_num_proc)
if eval_dataset is not None:
    eval_dataset = eval_dataset.map(fmt, num_proc=step_map_num_proc)

from collections import Counter


def _validate_action_supervision_rows(ds, label, mode, sample_limit=512):
    if ds is None or len(ds) == 0:
        print(f'{label} supervision check skipped: empty dataset')
        return
    if mode not in {'action_only', 'action_reason'}:
        print(f'{label} supervision check skipped for mode={mode}')
        return

    limit = min(int(sample_limit), len(ds))
    feasible_len_counter = Counter()
    matched = 0
    for i in range(limit):
        row = ds[i]
        action_positions = [idx for idx, flag in enumerate(row.get('action_target_mask', [])) if int(flag) == 1]
        if len(action_positions) != 1:
            raise ValueError(f'{label} row {i}: expected exactly one action target position, got {len(action_positions)}')
        pos = int(action_positions[0])
        labels_row = row.get('labels', [])
        feasible_ids = [int(x) for x in row.get('feasible_action_ids', [])]
        if pos >= len(labels_row):
            raise ValueError(f'{label} row {i}: action position {pos} exceeds labels length {len(labels_row)}')
        target_id = int(labels_row[pos])
        if target_id == -100:
            raise ValueError(f'{label} row {i}: action target label is -100 at position {pos}')
        if target_id not in feasible_ids:
            raise ValueError(
                f'{label} row {i}: target_id {target_id} not found in feasible_action_ids '
                f'(count={len(feasible_ids)}, feasible_head={feasible_ids[:8]})'
            )
        feasible_len_counter[len(feasible_ids)] += 1
        matched += 1

    print(f'{label} action supervision check:', {
        'checked_rows': limit,
        'matched_targets': matched,
        'feasible_count_dist_top': feasible_len_counter.most_common(10),
        'single_feasible_ratio': (feasible_len_counter.get(1, 0) / limit) if limit else 0.0,
    })


_validate_action_supervision_rows(train_dataset, 'train', resolved_step_supervision_mode)
if eval_dataset is not None:
    _validate_action_supervision_rows(eval_dataset, 'eval', resolved_step_supervision_mode)

enable_eval = bool(CFG.get('enable_eval', True))
eval_strategy_value = CFG['evaluation_strategy'] if enable_eval else 'no'
eval_steps_value = CFG['eval_steps'] if enable_eval else None
eval_dataset_for_trainer = eval_dataset if enable_eval else None

print('train rows:', len(train_dataset))
print('eval rows:', len(eval_dataset) if eval_dataset is not None else 0)
if len(train_dataset):
    sample_feature_schema = train_dataset[0].get('feature_schema_version', 'unknown') if hasattr(train_dataset[0], 'get') else 'unknown'
    print('feature schema:', sample_feature_schema)
    print('sample prompt\n', train_dataset[0]['text'][:500])
    print('sample supervision stats', {
        'prompt_tokens': train_dataset[0].get('prompt_token_count'),
        'assistant_tokens': train_dataset[0].get('assistant_token_count'),
        'supervised_tokens': train_dataset[0].get('supervised_token_count'),
    })

if len(train_dataset):
    sample_row = train_dataset[0]
    print('sample row keys:', list(sample_row.keys()))
    print('\n[sample state_text]')
    print(str(sample_row.get('state_text', ''))[:3000])
    print('\n[sample target_text]')
    print(sample_row.get('target_text', ''))
    print('\n[sample input_ids length]')
    print(len(sample_row.get('input_ids', [])))
    label_positions = [i for i, x in enumerate(sample_row.get('labels', [])) if int(x) != -100]
    print('\n[sample labels non -100 positions]')
    print(label_positions[:50], '... total =', len(label_positions))
    action_positions = [i for i, x in enumerate(sample_row.get('action_target_mask', [])) if int(x) == 1]
    print('\n[sample action_target_mask positions]')
    print(action_positions)
    print('\n[sample action_codes]')
    print(sample_row.get('action_codes', []))
    print('\n[sample raw_feasible_action_codes]')
    print(sample_row.get('raw_feasible_action_codes', []))
    print('\n[sample feasible_action_ids]')
    print(sample_row.get('feasible_action_ids', []))
    print('\n[sample feasible_action_tokens]')
    print([tokenizer.decode([int(x)], skip_special_tokens=False) for x in sample_row.get('feasible_action_ids', [])])
    if action_positions:
        pos = int(action_positions[0])
        label_id = int(sample_row['labels'][pos])
        print('\n[sample action target detail]')
        print({'pos': pos, 'label_id': label_id, 'decoded': tokenizer.decode([label_id], skip_special_tokens=False)})
        print('target_in_feasible =', label_id in [int(x) for x in sample_row.get('feasible_action_ids', [])])
        print('sample action loss_weight =', float(sample_row.get('loss_weights', [0.0] * (pos + 1))[pos]))
    print('\n[sample nonzero loss_weights positions]')
    nz_weights = [(i, float(x)) for i, x in enumerate(sample_row.get('loss_weights', [])) if float(x) != 0.0]
    print(nz_weights[:20], '... total =', len(nz_weights))
    print('\n[sample prompt/assistant/supervised counts]')
    print({
        'prompt_token_count': sample_row.get('prompt_token_count'),
        'assistant_token_count': sample_row.get('assistant_token_count'),
        'supervised_token_count': sample_row.get('supervised_token_count'),
    })

print('dataset preprocessing ready; run the next cell to start training')





base model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
adapter_role: mixed
resolved step supervision mode: action_reason


train_data/jssp_step_train_with_reason_d(…):   0%|          | 0.00/47.3G [00:00<?, ?B/s]

step dataset: /root/.cache/huggingface/hub/datasets--HYUNJINI--jssp_mixed_step_train_dispatch_v1/snapshots/786187c593d6256652713191616e0f2e74dad014/train_data/jssp_step_train_with_reason_dispatch.jsonl
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

raw dataset rows: 2,130,560
raw feasible_count_dist_top: [(2, 991847), (3, 516688), (4, 260640), (5, 137037), (6, 80882), (7, 53172), (8, 38211), (9, 28964), (10, 23119)]
raw feasible_count>=2 rows: 2130560
raw feasible_count>=2 ratio: 1.0
raw feasible_count>=3 rows: 1138713
raw feasible_count>=3 ratio: 0.5344665252328027
raw feasible filter kept rows: 2,130,560/4,125,000
raw feasible filter kept ratio: 0.5164993939393939
raw feasible filter dropped rows: 1994440
raw feasible filter dropped ratio: 0.4835006060606061
instance split: mode=fixed_per_size train_instances=112,500 eval_instances=0 overlap=0
row counts before map: train_rows=2,130,560 eval_rows=0
train feasible_count_dist_top (before feasible filter): [(2, 991847), (3, 516688), (4, 260640), (5, 137037), (6, 80882), (7, 53172), (8, 38211), (9, 28964), (10, 23119)]
train feasible_count>=2 rows (before feasible filter): 2130560
train feasible_count>=2 ratio (before feasible filter): 1.0
train feasible_count>=3 rows (before feasi

Map (num_proc=16):   0%|          | 0/2130560 [00:00<?, ? examples/s]

train action supervision check: {'checked_rows': 512, 'matched_targets': 512, 'feasible_count_dist_top': [(2, 242), (3, 109), (4, 75), (5, 33), (7, 13), (6, 13), (9, 12), (10, 8), (8, 7)], 'single_feasible_ratio': 0.0}
train rows: 2130560
eval rows: 0
feature schema: jssp_step_dispatch_v1_transition_action_token
sample prompt
 
sample supervision stats {'prompt_tokens': 578, 'assistant_tokens': 203, 'supervised_tokens': 203}
sample row keys: ['instance_id', 'source_index', 'state_text', 'reason_input_text', 'target_text', 'target_action_reason_text', 'target_reason_text', 'reason_target_text', 'feature_schema_version', 'num_jobs', 'num_machines', 'total_steps', 'step_idx', 'action_codes', 'feasible_action_codes', 'num_feasible_actions', 'action_code_to_job', 'raw_feasible_action_codes', 'text', 'input_ids', 'attention_mask', 'labels', 'loss_weights', 'action_target_mask', 'feasible_action_ids', 'prompt_token_count', 'assistant_token_count', 'supervised_token_count']

[sample state_text

#### 이거는 데이터 max_length를 위한 테스트용

In [ ]:
print('[dataset length scan start]')
print('train rows:', len(train_dataset))

longest_row = None
longest_10x10_first_step = None

for row_idx in range(len(train_dataset)):
    row = train_dataset[row_idx]
    item = {
        'row_idx': int(row_idx),
        'instance_id': str(row.get('instance_id', '')),
        'num_jobs': int(row.get('num_jobs', 0) or 0),
        'num_machines': int(row.get('num_machines', 0) or 0),
        'step_idx': int(row.get('step_idx', -1) or -1),
        'seq_len': int(len(row.get('input_ids', []))),
        'prompt_tokens': int(row.get('prompt_token_count', 0) or 0),
        'assistant_tokens': int(row.get('assistant_token_count', 0) or 0),
        'supervised_tokens': int(row.get('supervised_token_count', 0) or 0),
        'feasible_actions': int(row.get('num_feasible_actions', len(row.get('action_codes', [])))),
    }
    if longest_row is None or item['seq_len'] > longest_row['seq_len']:
        longest_row = item
    if item['num_jobs'] == 10 and item['num_machines'] == 10 and item['step_idx'] == 0:
        if longest_10x10_first_step is None or item['seq_len'] > longest_10x10_first_step['seq_len']:
            longest_10x10_first_step = item

print('\n[longest sequence in current train_dataset]')
if longest_row is None:
    print('train_dataset is empty')
else:
    print(longest_row)
    print('recommended_max_seq_length_min:', longest_row['seq_len'] + 32)

print('[longest 10x10 first-step sequence in current train_dataset]')
if longest_10x10_first_step is None:
    print('No 10x10 first-step row found in current train_dataset.')
else:
    print(longest_10x10_first_step)
    print('recommended_max_seq_length_min_for_10x10_first_step:', longest_10x10_first_step['seq_len'] + 32)


[dataset length scan start]
train rows: 2130560

[longest sequence in current train_dataset]
{'row_idx': 1948464, 'instance_id': 'train_111716', 'num_jobs': 10, 'num_machines': 10, 'step_idx': -1, 'seq_len': 2461, 'prompt_tokens': 1709, 'assistant_tokens': 752, 'supervised_tokens': 752, 'feasible_actions': 10}
recommended_max_seq_length_min: 2493
[longest 10x10 first-step sequence in current train_dataset]
No 10x10 first-step row found in current train_dataset.


#### 이게 학습

In [ ]:
print('starting training cell; re-run previous cell if you changed dataset/preprocessing settings or want a fresh model state')
import inspect
import math
import torch.nn as nn

LOSS_MIN_FEASIBLE_ACTIONS = max(2, int(CFG.get('min_train_feasible_actions', 1)))
print('loss min feasible actions:', LOSS_MIN_FEASIBLE_ACTIONS)

class StepSupervisionCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, features):
        max_len = max(len(feature['input_ids']) for feature in features)
        max_action_count = max(len(feature.get('feasible_action_ids', [])) for feature in features)
        batch = {
            'input_ids': [],
            'attention_mask': [],
            'labels': [],
            'loss_weights': [],
            'action_target_mask': [],
            'feasible_action_ids': [],
        }
        for feature in features:
            seq_len = len(feature['input_ids'])
            pad_len = max_len - seq_len
            batch['input_ids'].append(list(feature['input_ids']) + [int(self.pad_token_id)] * pad_len)
            batch['attention_mask'].append(list(feature['attention_mask']) + [0] * pad_len)
            batch['labels'].append(list(feature['labels']) + [-100] * pad_len)
            batch['loss_weights'].append(list(feature['loss_weights']) + [0.0] * pad_len)
            batch['action_target_mask'].append(list(feature.get('action_target_mask', [])) + [0] * pad_len)
            feasible_action_ids = list(feature.get('feasible_action_ids', []))
            batch['feasible_action_ids'].append(feasible_action_ids + [-1] * (max_action_count - len(feasible_action_ids)))
        return {
            'input_ids': torch.tensor(batch['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(batch['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(batch['labels'], dtype=torch.long),
            'loss_weights': torch.tensor(batch['loss_weights'], dtype=torch.float32),
            'action_target_mask': torch.tensor(batch['action_target_mask'], dtype=torch.long),
            'feasible_action_ids': torch.tensor(batch['feasible_action_ids'], dtype=torch.long),
        }


def _build_step_supervision_trainer(base_cls):
    class StepSupervisionTrainer(base_cls):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            model_inputs = dict(inputs)
            labels = model_inputs.pop('labels')
            loss_weights = model_inputs.pop('loss_weights', None)
            action_target_mask = model_inputs.pop('action_target_mask', None)
            feasible_action_ids = model_inputs.pop('feasible_action_ids', None)
            outputs = model(**model_inputs)
            logits = outputs.get('logits') if isinstance(outputs, dict) else outputs.logits

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss_fct = nn.CrossEntropyLoss(ignore_index=-100, reduction='none')
            token_loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            ).view_as(shift_labels)

            valid_mask = shift_labels.ne(-100)
            if loss_weights is None:
                weights = valid_mask.to(token_loss.dtype)
            else:
                shift_weights = loss_weights[..., 1:].contiguous().to(token_loss.dtype)
                weights = shift_weights * valid_mask.to(token_loss.dtype)

            if action_target_mask is not None:
                shift_action_mask = action_target_mask[..., 1:].contiguous().bool()
            else:
                shift_action_mask = torch.zeros_like(valid_mask, dtype=torch.bool)

            base_weights = weights * (~shift_action_mask).to(weights.dtype)
            base_loss_num = (token_loss * base_weights).sum()
            base_denom = base_weights.sum()

            filtered_loss_num = base_loss_num
            filtered_denom = base_denom
            all_loss_num = base_loss_num
            all_denom = base_denom

            matched_action_targets = 0
            matched_action_targets_all = 0
            if feasible_action_ids is not None and bool(shift_action_mask.any().item()):
                action_rows, action_cols = torch.nonzero(shift_action_mask, as_tuple=True)
                for row_idx, col_idx in zip(action_rows.tolist(), action_cols.tolist()):
                    row_feasible_ids = feasible_action_ids[row_idx]
                    row_feasible_ids = row_feasible_ids[row_feasible_ids.ge(0)]
                    if row_feasible_ids.numel() <= 0:
                        raise ValueError(
                            "Empty feasible_action_ids encountered at an action-supervised position."
                        )
                    target_id = int(shift_labels[row_idx, col_idx].item())
                    target_matches = torch.nonzero(row_feasible_ids.eq(target_id), as_tuple=False).view(-1)
                    if target_matches.numel() <= 0:
                        raise ValueError(
                            "Action target token id was not found inside feasible_action_ids. "
                            f"target_id={target_id}, feasible_count={int(row_feasible_ids.numel())}, "
                            f"feasible_head={row_feasible_ids[:8].tolist()}"
                        )
                    candidate_logits = shift_logits[row_idx, col_idx].index_select(
                        0,
                        row_feasible_ids.to(shift_logits.device, dtype=torch.long),
                    )
                    target_index = target_matches[0].to(device=shift_logits.device, dtype=torch.long).view(1)
                    action_loss = nn.functional.cross_entropy(
                        candidate_logits.view(1, -1),
                        target_index,
                        reduction='sum',
                    )
                    action_weight = weights[row_idx, col_idx].to(action_loss.dtype)

                    all_loss_num = all_loss_num + (action_loss * action_weight)
                    all_denom = all_denom + action_weight
                    matched_action_targets_all += 1

                    if row_feasible_ids.numel() < LOSS_MIN_FEASIBLE_ACTIONS:
                        continue

                    filtered_loss_num = filtered_loss_num + (action_loss * action_weight)
                    filtered_denom = filtered_denom + action_weight
                    matched_action_targets += 1

                if matched_action_targets <= 0:
                    raise ValueError(
                        f"No matched action targets with feasible_count >= {LOSS_MIN_FEASIBLE_ACTIONS} contributed to the action-centric loss."
                    )

            all_denom_safe = all_denom.clamp_min(1.0)
            filtered_denom_safe = filtered_denom.clamp_min(1.0)
            loss_all = all_loss_num / all_denom_safe
            loss_filtered = filtered_loss_num / filtered_denom_safe

            self._last_loss_all_feasible = float(loss_all.detach().float().cpu().item())
            self._last_loss_filtered_feasible = float(loss_filtered.detach().float().cpu().item())
            self._last_action_targets_all = int(matched_action_targets_all)
            self._last_action_targets_filtered = int(matched_action_targets)

            return (loss_filtered, outputs) if return_outputs else loss_filtered

        def log(self, logs, *args, **kwargs):
            logs = dict(logs)
            if 'loss' in logs and hasattr(self, '_last_loss_all_feasible'):
                logs.setdefault('loss_all_feasible', round(float(self._last_loss_all_feasible), 6))
                logs.setdefault(f'loss_feasible_ge{LOSS_MIN_FEASIBLE_ACTIONS}', round(float(self._last_loss_filtered_feasible), 6))
                logs.setdefault('action_targets_all', int(getattr(self, '_last_action_targets_all', 0)))
                logs.setdefault(f'action_targets_ge{LOSS_MIN_FEASIBLE_ACTIONS}', int(getattr(self, '_last_action_targets_filtered', 0)))
            return super().log(logs, *args, **kwargs)

    return StepSupervisionTrainer


def _debug_manual_action_loss(trainer, model, batch, tokenizer, max_rows=8):
    print('\n[DEBUG manual batch keys]', sorted(batch.keys()))
    device_batch = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    model_inputs = {k: v for k, v in device_batch.items() if k not in {'labels', 'loss_weights', 'action_target_mask', 'feasible_action_ids'}}
    labels = device_batch['labels']
    loss_weights = device_batch['loss_weights']
    action_target_mask = device_batch['action_target_mask']
    feasible_action_ids = device_batch['feasible_action_ids']

    model.eval()
    with torch.no_grad():
        outputs = model(**model_inputs)
        logits = outputs.get('logits') if isinstance(outputs, dict) else outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        shift_weights = loss_weights[..., 1:].contiguous().to(torch.float32)
        shift_action_mask = action_target_mask[..., 1:].contiguous().bool()

        print('[DEBUG tensor shapes]', {
            'input_ids': tuple(device_batch['input_ids'].shape),
            'labels': tuple(labels.shape),
            'shift_logits': tuple(shift_logits.shape),
            'shift_labels': tuple(shift_labels.shape),
            'shift_weights': tuple(shift_weights.shape),
            'shift_action_mask_true': int(shift_action_mask.sum().item()),
            'feasible_action_ids': tuple(feasible_action_ids.shape),
        })

        per_row = []
        action_rows, action_cols = torch.nonzero(shift_action_mask, as_tuple=True)
        for idx, (row_idx, col_idx) in enumerate(zip(action_rows.tolist(), action_cols.tolist())):
            if idx >= int(max_rows):
                break
            target_id = int(shift_labels[row_idx, col_idx].item())
            weight = float(shift_weights[row_idx, col_idx].item())
            row_feasible_ids = feasible_action_ids[row_idx]
            row_feasible_ids = row_feasible_ids[row_feasible_ids.ge(0)]
            feasible_ids = [int(x) for x in row_feasible_ids.tolist()]
            candidate_logits = shift_logits[row_idx, col_idx].index_select(0, row_feasible_ids.to(shift_logits.device, dtype=torch.long))
            included_in_loss = len(feasible_ids) >= int(LOSS_MIN_FEASIBLE_ACTIONS)
            if target_id in feasible_ids:
                target_index = feasible_ids.index(target_id)
                ce = torch.nn.functional.cross_entropy(candidate_logits.view(1, -1), torch.tensor([target_index], device=candidate_logits.device), reduction='sum')
                ce_value = float(ce.detach().cpu().item())
                ce_finite = bool(torch.isfinite(ce).item())
            else:
                target_index = None
                ce_value = None
                ce_finite = None
            probs = torch.softmax(candidate_logits.float(), dim=-1)
            topk = min(5, int(probs.numel()))
            top_vals, top_idx = torch.topk(probs, k=topk)
            top_tokens = []
            for p, j in zip(top_vals.tolist(), top_idx.tolist()):
                tok_id = feasible_ids[int(j)]
                top_tokens.append((tokenizer.decode([tok_id], skip_special_tokens=False), float(p)))
            per_row.append({
                'row_idx': int(row_idx),
                'col_idx': int(col_idx),
                'target_id': target_id,
                'target_token': tokenizer.decode([target_id], skip_special_tokens=False),
                'weight': weight,
                'feasible_count': len(feasible_ids),
                'included_in_loss': included_in_loss,
                'feasible_tokens_head': [tokenizer.decode([x], skip_special_tokens=False) for x in feasible_ids[:10]],
                'target_in_feasible': target_id in feasible_ids,
                'target_index': target_index,
                'candidate_logits_min': float(candidate_logits.min().detach().cpu().item()),
                'candidate_logits_max': float(candidate_logits.max().detach().cpu().item()),
                'candidate_logits_mean': float(candidate_logits.float().mean().detach().cpu().item()),
                'ce_value': ce_value,
                'ce_isfinite': ce_finite,
                'top_probs': top_tokens,
            })

        print('[DEBUG action rows head]')
        for item in per_row:
            print(item)

        manual_loss = trainer.compute_loss(model, device_batch)
        print('[DEBUG manual trainer.compute_loss (train/ge-filtered)]', manual_loss)
        print('[DEBUG manual logged loss_all_feasible]', getattr(trainer, '_last_loss_all_feasible', None))
        print('[DEBUG manual logged loss_filtered_feasible]', getattr(trainer, '_last_loss_filtered_feasible', None))
        print('[DEBUG manual loss finite]', bool(torch.isfinite(manual_loss).item()))
        if not bool(torch.isfinite(manual_loss).item()):
            print('[DEBUG manual loss value raw]', manual_loss.detach().cpu())

    model.train()

if CFG['enable_wandb']:
    project_name = CFG['wandb_project'] or f"{task_type}_optimization"
    wandb.init(project=project_name, name=f"{task_type}_{CFG['model_type']}_r{CFG['lora_r']}", config=CFG)
else:
    os.environ['WANDB_MODE'] = 'disabled'

with open(output_dir / 'training_hyperparams_args.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for k, v in CFG.items():
        writer.writerow([k, v])

if CFG['train_lm_head'] and CFG['train_embed_tokens']:
    from unsloth import UnslothTrainer, UnslothTrainingArguments
    Trainer = UnslothTrainer
    TrainingArgs = UnslothTrainingArguments
    print('Training with UnslothTrainer')
else:
    from transformers import Trainer as HFTrainer, TrainingArguments
    Trainer = HFTrainer
    TrainingArgs = TrainingArguments
    print('Training with Trainer')

Trainer = _build_step_supervision_trainer(Trainer)
data_collator = StepSupervisionCollator(tokenizer=tokenizer)

effective_bf16 = CFG['bf16'] if CFG['bf16'] is not None else (True if CFG['dtype'] == 'bfloat16' else False)
effective_fp16 = CFG['fp16'] if CFG['fp16'] is not None else (True if CFG['dtype'] == 'float16' else False)

training_args_kwargs = {
    'per_device_train_batch_size': CFG['per_device_train_batch_size'],
    'gradient_accumulation_steps': CFG['gradient_accumulation_steps'],
    'warmup_steps': CFG['warmup_steps'],
    'num_train_epochs': CFG['num_train_epochs'],
    'learning_rate': CFG['learning_rate'],
    'bf16': effective_bf16,
    'fp16': effective_fp16,
    'logging_steps': CFG['logging_steps'],
    'logging_first_step': True,
    'logging_nan_inf_filter': bool(CFG.get('logging_nan_inf_filter', False)),
    'optim': CFG['optim'],
    'weight_decay': CFG['weight_decay'],
    'lr_scheduler_type': CFG['lr_scheduler_type'],
    'seed': CFG['seed'],
    'output_dir': str(output_dir),
    'report_to': (CFG['report_to'] if CFG['enable_wandb'] else 'none'),
    'load_best_model_at_end': (CFG['load_best_model_at_end'] if enable_eval else False),
    'metric_for_best_model': CFG['metric_for_best_model'],
    'greater_is_better': CFG['greater_is_better'],
    'save_total_limit': CFG['save_total_limit'],
    'save_steps': CFG['save_steps'],
    'save_strategy': CFG['save_strategy'],
    'eval_strategy': eval_strategy_value,
    'eval_steps': eval_steps_value,
    'per_device_eval_batch_size': CFG['per_device_eval_batch_size'],
    'group_by_length': CFG['group_by_length'],
    'dataloader_num_workers': CFG['dataloader_num_workers'],
    'remove_unused_columns': CFG['remove_unused_columns'],
    'run_name': CFG['run_name'],
}

supported_training_arg_names = {
    name for name in inspect.signature(TrainingArgs.__init__).parameters.keys()
    if name != 'self'
}

if 'eval_strategy' in training_args_kwargs and 'evaluation_strategy' in supported_training_arg_names:
    training_args_kwargs['evaluation_strategy'] = training_args_kwargs.pop('eval_strategy')
elif 'evaluation_strategy' in training_args_kwargs and 'eval_strategy' in supported_training_arg_names:
    training_args_kwargs['eval_strategy'] = training_args_kwargs.pop('evaluation_strategy')

final_training_args = {
    k: v for k, v in training_args_kwargs.items()
    if v is not None and k in supported_training_arg_names
}
dropped_training_args = sorted(
    k for k, v in training_args_kwargs.items()
    if v is not None and k not in supported_training_arg_names
)
if dropped_training_args:
    print('TrainingArgs unsupported keys dropped:', dropped_training_args)
print('resolved training args subset:', {
    'per_device_train_batch_size': final_training_args.get('per_device_train_batch_size'),
    'gradient_accumulation_steps': final_training_args.get('gradient_accumulation_steps'),
    'per_device_eval_batch_size': final_training_args.get('per_device_eval_batch_size'),
    'num_train_epochs': final_training_args.get('num_train_epochs'),
    'learning_rate': final_training_args.get('learning_rate'),
    'save_steps': final_training_args.get('save_steps'),
})

callbacks = []
if CFG.get('upload_on_save', False):
    callbacks.append(
        UploadCheckpointToHFCallback(
            repo_id=CFG['hf_model_repo_out'],
            token=HF_TOKEN,
            output_dir=output_dir,
            upload_source=CFG.get('upload_source', 'latest_checkpoint'),
        )
    )
    print(f"checkpoint auto-upload enabled -> {CFG['hf_model_repo_out']}")
else:
    print('checkpoint auto-upload disabled')

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_trainer,
    data_collator=data_collator,
    args=TrainingArgs(**final_training_args),
    callbacks=callbacks if callbacks else None,
)
print('🎯 action-centric supervision:', {
    'mode': resolved_step_supervision_mode,
    'action_loss_weight': float(CFG.get('action_loss_weight', 4.0)),
    'supervision': 'assistant-only/action-targeted+feasible-set-ce',
})
print('DEBUG: trainer object created')
print('DEBUG: trainer args logging_nan_inf_filter =', getattr(trainer.args, 'logging_nan_inf_filter', None))
print('DEBUG: trainer args group_by_length =', getattr(trainer.args, 'group_by_length', None))
print('DEBUG: trainer args batch_size =', getattr(trainer.args, 'per_device_train_batch_size', None))

debug_features = []
for _i in range(min(8, len(train_dataset))):
    debug_features.append(train_dataset[_i])
if debug_features:
    debug_batch = data_collator(debug_features)
    print('DEBUG: collated debug batch shapes', {k: tuple(v.shape) for k, v in debug_batch.items() if hasattr(v, 'shape')})
    print('DEBUG: debug batch action counts', [int((row > -1).sum().item()) for row in debug_batch['feasible_action_ids']])
    print('DEBUG: debug batch action target counts', [int(row.sum().item()) for row in debug_batch['action_target_mask']])
    _debug_manual_action_loss(trainer, model, debug_batch, tokenizer, max_rows=8)

resume_arg = None

if CFG['resume_from_checkpoint']:
    checkpoint_path = os.path.expanduser(CFG['resume_from_checkpoint'])
    if os.path.exists(checkpoint_path):
        print('resume from checkpoint:', checkpoint_path)
        resume_arg = checkpoint_path
    else:
        print('checkpoint not found, train from scratch:', checkpoint_path)
else:
    contains_checkpoints = any('checkpoint' in p.name and p.is_dir() for p in output_dir.iterdir())
    if contains_checkpoints:
        print('checkpoint detected in output_dir, continue training')
        resume_arg = True
    else:
        print('no checkpoint in output_dir, train from scratch')

print('DEBUG: entering trainer.train()', 'resume_arg=', resume_arg, flush=True)

if resume_arg is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=resume_arg)

final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

print('training done')
print('output_dir:', output_dir)
print('final_dir:', final_dir)
print('checkpoints:')
for p in sorted(output_dir.glob('checkpoint-*')):
    print(' -', p)



starting training cell; re-run previous cell if you changed dataset/preprocessing settings or want a fresh model state
loss min feasible actions: 2
Training with Trainer
resolved training args subset: {'per_device_train_batch_size': 8, 'gradient_accumulation_steps': 2, 'per_device_eval_batch_size': 8, 'num_train_epochs': 2, 'learning_rate': 0.0002, 'save_steps': 1000}
checkpoint auto-upload enabled -> HYUNJINI/jssp_policy_llama8b_step_r64_ep2
🎯 action-centric supervision: {'mode': 'action_reason', 'action_loss_weight': 4.0, 'supervision': 'assistant-only/action-targeted+feasible-set-ce'}
DEBUG: trainer object created
DEBUG: trainer args logging_nan_inf_filter = False
DEBUG: trainer args group_by_length = False
DEBUG: trainer args batch_size = 8
DEBUG: collated debug batch shapes {'input_ids': (8, 2438), 'attention_mask': (8, 2438), 'labels': (8, 2438), 'loss_weights': (8, 2438), 'action_target_mask': (8, 2438), 'feasible_action_ids': (8, 10)}
DEBUG: debug batch action counts [2, 2, 2, 

/tmp/ipykernel_759/1387814345.py:350: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `StepSupervisionTrainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(


[DEBUG tensor shapes] {'input_ids': (8, 2438), 'labels': (8, 2438), 'shift_logits': (8, 2437, 138255), 'shift_labels': (8, 2437), 'shift_weights': (8, 2437), 'shift_action_mask_true': 8, 'feasible_action_ids': (8, 10)}
[DEBUG action rows head]
{'row_idx': 0, 'col_idx': 577, 'target_id': 135307, 'target_token': '<A7052>', 'weight': 4.0, 'feasible_count': 2, 'included_in_loss': True, 'feasible_tokens_head': ['<A5720>', '<A7052>'], 'target_in_feasible': True, 'target_index': 1, 'candidate_logits_min': 1.0546875, 'candidate_logits_max': 1.0546875, 'candidate_logits_mean': 1.0546875, 'ce_value': 0.69140625, 'ce_isfinite': True, 'top_probs': [('<A7052>', 0.5), ('<A5720>', 0.5)]}
{'row_idx': 1, 'col_idx': 572, 'target_id': 134609, 'target_token': '<A6354>', 'weight': 4.0, 'feasible_count': 2, 'included_in_loss': True, 'feasible_tokens_head': ['<A6354>', '<A8450>'], 'target_in_feasible': True, 'target_index': 0, 'candidate_logits_min': 1.09375, 'candidate_logits_max': 1.09375, 'candidate_logit

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,130,560 | Num Epochs = 2 | Total steps = 266,320
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 335,544,320 of 8,447,717,376 (3.97% trained)


Unsloth: Will smartly offload gradients to save VRAM!


...

  ...oint-30000/tokenizer.json: 100%|##########| 19.0MB / 19.0MB            

  ...adapter_model.safetensors:   0%|          | 52.5kB / 3.61GB            

  ...kpoint-30000/optimizer.pt:   0%|          |  529kB /  683MB            

  ...kpoint-30000/scheduler.pt:  10%|9         |   141B / 1.47kB            

UploadCheckpointToHFCallback: uploaded /content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128/checkpoint-30000 -> HYUNJINI/jssp_policy_llama8b_step_r64_ep2


## 4) (선택) 학습 완료 모델 HF 업로드


#### 모델 drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

src_output_dir = Path(output_dir)  # colab_02에서 이미 정의됨
dst_root = Path('/content/drive/MyDrive/LLM_JSSP_checkpoints') / src_output_dir.name
dst_root.mkdir(parents=True, exist_ok=True)

checkpoints = sorted(
    [p for p in src_output_dir.glob('checkpoint-*') if p.is_dir()],
    key=lambda p: int(p.name.split('-')[-1])
)

if not checkpoints:
    print('No checkpoint folder found in output_dir:', src_output_dir)
else:
    latest_ckpt = checkpoints[-1]
    dst_ckpt = dst_root / latest_ckpt.name

    if dst_ckpt.exists():
        shutil.rmtree(dst_ckpt)
    shutil.copytree(latest_ckpt, dst_ckpt)

    print('copied latest checkpoint to drive:')
    print('src =', latest_ckpt)
    print('dst =', dst_ckpt)
# CFG['resume_from_checkpoint'] = '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_policy_dispatch_llama8b_r64/checkpoint-9000'

Mounted at /content/drive
copied latest checkpoint to drive:
src = outputs/jssp_mixed_dispatch_llama8b_r128/checkpoint-9000
dst = /content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128/checkpoint-9000


#### 데이터 drive

In [ ]:
from pathlib import Path
import shutil
import json

if 'step_dataset_path' not in globals() or not step_dataset_path:
    raise RuntimeError('step_dataset_path is not defined. Run the dataset/preprocessing cell first.')

src_dataset = Path(step_dataset_path)
if not src_dataset.exists():
    raise FileNotFoundError(f'dataset file not found: {src_dataset}')

backup_root = Path('/content/drive/MyDrive/LLM_JSSP_datasets')
if 'output_dir' in globals() and output_dir is not None:
    backup_root = backup_root / Path(output_dir).name
backup_root.mkdir(parents=True, exist_ok=True)

dst_dataset = backup_root / src_dataset.name
shutil.copy2(src_dataset, dst_dataset)

manifest = {
    'src_dataset': str(src_dataset),
    'dst_dataset': str(dst_dataset),
    'dataset_size_bytes': src_dataset.stat().st_size,
    'adapter_role': CFG.get('adapter_role'),
    'env_mode': CFG.get('env_mode'),
    'step_supervision_mode': CFG.get('step_supervision_mode'),
    'dataset_source': CFG.get('dataset_source'),
    'step_dataset_hf_repo': CFG.get('step_dataset_hf_repo'),
    'step_dataset_hf_file': CFG.get('step_dataset_hf_file'),
    'policy_step_dataset_hf_repo_dispatch': CFG.get('policy_step_dataset_hf_repo_dispatch'),
    'policy_step_dataset_hf_file_dispatch': CFG.get('policy_step_dataset_hf_file_dispatch'),
    'reason_step_dataset_hf_repo_dispatch': CFG.get('reason_step_dataset_hf_repo_dispatch'),
    'reason_step_dataset_hf_file_dispatch': CFG.get('reason_step_dataset_hf_file_dispatch'),
    'mixed_step_dataset_hf_repo_dispatch': CFG.get('mixed_step_dataset_hf_repo_dispatch'),
    'mixed_step_dataset_hf_file_dispatch': CFG.get('mixed_step_dataset_hf_file_dispatch'),
}
manifest_path = backup_root / 'dataset_backup_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('dataset copied to drive:')
print('src =', src_dataset)
print('dst =', dst_dataset)
print('manifest =', manifest_path)


OSError: [Errno 28] No space left on device: '/root/.cache/huggingface/hub/datasets--HYUNJINI--jssp_mixed_step_train_dispatch_v1/snapshots/786187c593d6256652713191616e0f2e74dad014/train_data/jssp_step_train_with_reason_dispatch.jsonl' -> '/content/drive/MyDrive/LLM_JSSP_datasets/jssp_mixed_dispatch_llama8b_r128/jssp_step_train_with_reason_dispatch.jsonl'

#### HF에 모델 업로드

In [ ]:
if CFG['upload_after_train']:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=CFG['hf_model_repo_out'], repo_type='model', private=False, exist_ok=True)

    output_dir = Path(resolve_output_dir(CFG, resolve_task_type(CFG)))
    upload_dir = resolve_upload_dir(output_dir, CFG['upload_source'], CFG['checkpoint_tag'])
    print('upload_dir:', upload_dir)

    upload_folder(
        repo_id=CFG['hf_model_repo_out'],
        repo_type='model',
        folder_path=str(upload_dir),
        token=HF_TOKEN,
        commit_message=f"Upload LoRA ({upload_dir.name}) from colab_02_full",
    )

    files = api.list_repo_files(repo_id=CFG['hf_model_repo_out'], repo_type='model')
    print(f"upload done: {CFG['hf_model_repo_out']} ({len(files)} files)")
    for f in files:
        print(' -', f)
else:
    print('CFG[upload_after_train]=False, skip upload')
